# Production-Grade Chat System Design (HLD + LLD)

# 1. Requirements

## Functional Requirements

- 1:1 Chat
- Group Chat
- Online/Offline Status
- Read Receipts
- Typing Indicators
- Media Sharing
- Push Notifications
- Message Search
- Message Reactions
- Message Deletion
- Message Editing
- Message History

## Non Functional Requirements

- High Availability
- Low Latency (<100 ms)
- Scalability (100M+ users)
- Durability
- Fault Tolerance
- Security
- Multi-Region Support

---

# 2. Capacity Estimation

Assumptions:

- 100 Million DAU
- 10 Billion messages/day

Messages/sec:

10B / 86400 ≈ 115K msg/sec

Peak:

≈ 1 Million msg/sec

Average message size:

1 KB

Storage/day:

10 Billion × 1 KB ≈ 10 TB/day

---

# 3. High Level Architecture

```text
                +----------------+
                | Mobile/Web App |
                +--------+-------+
                         |
                    Load Balancer
                         |
       +-----------------+-----------------+
       |                 |                 |
+------+-----+   +------+-----+   +------+-----+
| Chat Server|   | Chat Server|   | Chat Server|
+------+-----+   +------+-----+   +------+-----+
       |                 |                 |
       +--------+--------+-----------------+
                |
          Message Bus
             Kafka
                |
      +---------+---------+
      |                   |
 Message Service   Notification Service
      |                   |
      v                   v
 Cassandra         APNs / FCM
      |
 Elasticsearch
```

---

# 4. Core Components

## API Gateway

Responsibilities:

- Authentication
- Rate Limiting
- Routing
- TLS Termination

---

## Chat Gateway

Maintains:

- WebSocket Connections
- Session State
- Presence

Responsibilities:

- Receive messages
- Push messages
- Typing indicators

---

## Message Service

Responsibilities:

- Validate message
- Persist message
- Publish to Kafka
- Generate sequence IDs

---

## Notification Service

Responsibilities:

- Offline notification delivery
- APNs integration
- FCM integration

---

## Presence Service

Tracks:

- Online
- Offline
- Last Seen

Stored in Redis.

---

# 5. Why WebSockets?

HTTP Polling:

```text
Client → Request → Server
```

WebSocket:

```text
Client <==== Persistent Connection ====> Server
```

Benefits:

- Real-time delivery
- Low latency
- Reduced overhead

---

# 6. User Login Flow

```text
User Login
    |
Authentication
    |
Connect WebSocket
    |
Register Session
    |
Store in Redis
```

Redis:

```text
user123 -> chat-server-7
```

---

# 7. Message Send Flow

```text
Sender
   |
Chat Server
   |
Kafka
   |
Message Service
   |
Cassandra
   |
Receiver Chat Server
   |
Receiver
```

---

# 8. Database Design

## Users

```sql
Users
-----
user_id
name
created_at
```

## Conversations

```sql
Conversation
------------
conversation_id
type
created_at
```

## Participants

```sql
Participants
------------
conversation_id
user_id
```

## Messages

```sql
Messages
--------
message_id
conversation_id
sender_id
sequence_no
content
timestamp
status
```

---

# 9. Message Storage

Use Cassandra.

Partition Key:

```text
conversation_id
```

Clustering Key:

```text
timestamp
```

Benefits:

- Horizontal scaling
- High write throughput

---

# 10. Sequence Numbers

Each conversation maintains:

```text
Conversation 101

1
2
3
4
5
```

Guarantees ordering.

Can use:

- Snowflake IDs
- Sequence Service

---

# 11. Message Ordering

Problem:

```text
M1
M2
M3
```

May arrive:

```text
M2
M1
M3
```

Solution:

Use sequence numbers.

Client sorts by:

```text
sequence_number
```

---

# 12. Kafka Usage

Topics:

```text
messages
notifications
presence
typing_events
```

Benefits:

- Durability
- Replay
- Backpressure handling

---

# 13. Presence Service

Stored in Redis.

Example:

```text
user123 -> ONLINE
```

Heartbeat every:

```text
30 sec
```

Offline detection:

```text
No heartbeat for 90 sec
```

---

# 14. Typing Indicator

Flow:

```text
User A typing
     |
Chat Server
     |
Redis Pub/Sub
     |
User B
```

Not persisted.

---

# 15. Read Receipts

States:

```text
SENT
DELIVERED
READ
```

Schema:

```sql
MessageReceipt
--------------
message_id
user_id
status
timestamp
```

---

# 16. Group Chat

Group:

```text
Group 1000
```

Members:

```text
A
B
C
D
```

Message Flow:

```text
Sender
  |
Kafka
  |
Fanout Service
  |
Recipients
```

---

# 17. Large Group Optimization

Avoid:

```text
1 write -> N writes
```

Store once:

```text
Conversation Store
```

Clients fetch messages.

Used by:

- WhatsApp Communities
- Telegram Channels

---

# 18. Push Notifications

User Offline

```text
Message
   |
Notification Service
   |
FCM / APNs
```

---

# 19. Search System

Use Elasticsearch.

Index:

```json
{
  "message":"hello world",
  "conversation":"123"
}
```

Supports:

- Keyword search
- Filters
- Date ranges

---

# 20. Media Messages

Store:

- Images
- Videos
- Documents

In Object Storage.

Example:

- S3
- GCS

Message stores:

```text
media_url
```

---

# 21. Security

Authentication:

- OAuth
- JWT

Encryption:

- TLS

Optional:

- End-to-End Encryption

---

# 22. End-to-End Encryption

Flow:

```text
Sender
Encrypt
   |
Server
Cannot Read
   |
Receiver
Decrypt
```

Examples:

- Signal
- WhatsApp

---

# 23. Multi Region Design

```text
US Region
EU Region
APAC Region
```

Local chat gateways.

Global Kafka replication.

Disaster Recovery enabled.

---

# 24. Redis Usage

Used for:

- Presence
- Session Mapping
- Rate Limiting
- Typing Indicators

Example:

```text
user -> server
```

---

# 25. Failure Handling

## Chat Server Crash

Reconnect.

Session restored.

## Kafka Failure

Replication.

## Database Failure

Multi-DC Cassandra.

---

# 26. Rate Limiting

Per User:

```text
100 msgs/min
```

Using:

- Redis Token Bucket

---

# 27. Observability

Metrics:

- Messages/sec
- Latency
- Failed deliveries
- Online users

Tools:

- Prometheus
- Grafana
- ELK

---

# 28. LLD Classes

```text
User
Conversation
Participant
Message
MessageReceipt
Presence
Notification
```

Message Class:

```java
class Message {
    String id;
    String conversationId;
    String senderId;
    String content;
    Long sequenceNumber;
    Long timestamp;
}
```

---

# 29. Scaling Strategy

Scale Independently:

- API Gateway
- Chat Gateway
- Kafka
- Message Service
- Notification Service

Partition Kafka by:

```text
conversation_id
```

Maintains ordering.

---

# 30. Interview Deep Dive Topics

- Why WebSockets?
- Why Kafka?
- Why Cassandra?
- Message Ordering
- Read Receipts
- Presence Tracking
- Group Chat Fanout
- End-to-End Encryption
- Multi Region Replication
- Redis Usage
- Failure Recovery
- Push Notification Design
- Search Architecture

This architecture can support hundreds of millions of users and billions of messages per day while maintaining low latency and high availability.
